In [20]:
#1. Import library
import pandas as pd
import numpy as np
from scipy import stats

In [21]:
#2. Import Data
df = pd.read_csv('Dataset/Spotify_Churn.csv')
df.head(100)

,user_id,gender,age,country,subscription_type,listening_time,songs_played_per_day,skip_rate,device_type,ads_listened_per_week,offline_listening,is_churned
0,1,Female,54,CA,Free,26,23,0.20,Desktop,31,0,1
1,2,Other,33,DE,Family,141,62,0.34,Web,0,1,0
2,3,Male,38,AU,Premium,199,38,0.04,Mobile,0,1,1
3,4,Female,22,CA,Student,36,2,0.31,Mobile,0,1,0
4,5,Other,29,US,Family,250,57,0.36,Mobile,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,Other,51,IN,Student,222,97,0.03,Desktop,0,1,1
96,97,Male,34,CA,Premium,196,69,0.07,Web,0,1,0
97,98,Female,47,US,Family,215,13,0.02,Mobile,0,1,1
98,99,Male,19,PK,Family,22,45,0.29,Web,0,1,1


In [22]:
#3. Inspect Data
#Menampilkan semua kolom
df.columns

Index(['user_id', 'gender', 'age', 'country', 'subscription_type',
       'listening_time', 'songs_played_per_day', 'skip_rate', 'device_type',
       'ads_listened_per_week', 'offline_listening', 'is_churned'],
      dtype='object')

In [23]:
#3. Inspect Data
#Memberikan informasi jenis tipe kolom
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                8000 non-null   int64  
 1   gender                 8000 non-null   object 
 2   age                    8000 non-null   int64  
 3   country                8000 non-null   object 
 4   subscription_type      8000 non-null   object 
 5   listening_time         8000 non-null   int64  
 6   songs_played_per_day   8000 non-null   int64  
 7   skip_rate              8000 non-null   float64
 8   device_type            8000 non-null   object 
 9   ads_listened_per_week  8000 non-null   int64  
 10  offline_listening      8000 non-null   int64  
 11  is_churned             8000 non-null   int64  
dtypes: float64(1), int64(7), object(4)
memory usage: 750.1+ KB


In [24]:
#3. Inspect Data
#Mendeteksi kolom duplikasi
df.duplicated().sum()

np.int64(0)

In [25]:
#3. Inspect Data
#Memisahkan data numerik dan kategorikal
nums = df.select_dtypes(include=['number']).columns.tolist() #nums = numeric
cats = df.select_dtypes(exclude=['number']).columns.tolist() #cats = kategorical

In [26]:
#3. Inspect Data
#Mendeskripsikan data
df.describe()

,user_id,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,is_churned
count,8000.00000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,4000.50000,37.662125,154.068250,50.127250,0.300127,6.943875,0.747750,0.258875
std,2309.54541,12.740359,84.015596,28.449762,0.173594,13.617953,0.434331,0.438044
min,1.00000,16.000000,10.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,2000.75000,26.000000,81.000000,25.000000,0.150000,0.000000,0.000000,0.000000
50%,4000.50000,38.000000,154.000000,50.000000,0.300000,0.000000,1.000000,0.000000
75%,6000.25000,49.000000,227.000000,75.000000,0.450000,5.000000,1.000000,1.000000
max,8000.00000,59.000000,299.000000,99.000000,0.600000,49.000000,1.000000,1.000000


In [27]:
#3. Inspect Data
#Mendeskripsikan data kategori
df.describe(include='object')

,gender,country,subscription_type,device_type
count,8000,8000,8000,8000
unique,3,8,4,3
top,Male,AU,Premium,Desktop
freq,2691,1034,2115,2778


In [28]:
#4. Handling missing value
#Melihat missing value
df.isna().sum()

user_id                  0
gender                   0
age                      0
country                  0
subscription_type        0
listening_time           0
songs_played_per_day     0
skip_rate                0
device_type              0
ads_listened_per_week    0
offline_listening        0
is_churned               0
dtype: int64

In [29]:
#4. Handling missing value
#Melihat jumlah kolom dan baris
df.shape

(8000, 12)

In [30]:
#6. Deteksi Outlier
#Mendeteksi dengan metode IQR 
Q1 = df[nums].quantile(0.25)
Q3 = df[nums].quantile(0.75)
IQR = Q3 - Q1

# Tentukan Batas Bawah dan Atas
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_outlier = df[((df[nums] < lower_bound) | (df[nums] > upper_bound)).any(axis=1)]
print('Data outlier: ', df_outlier.shape[0], 'baris')

Data outlier:  1683 baris


In [31]:
#6. Deteksi Outlier
#Hapus baris yang memiliki outlier di kolom numerik

df_no_outlier = df[~((df[nums] < lower_bound) | (df[nums] > upper_bound)).any(axis=1)]

print('Data sebelum: ', df.shape[0], 'baris')
print('Data sesudah: ', df_no_outlier.shape[0], 'baris')

Data sebelum:  8000 baris
Data sesudah:  6317 baris


In [32]:
#7. Feature Engineering

#Buat kategori usia 
df_no_outlier['age_group'] = pd.cut(df['age'], bins=[0,25,50,70], labels=['Teen','Adult','Old'])

#Buat kategori usia 
df_no_outlier['listening_group'] = pd.cut(df['listening_time'], bins=[0,100,200,300], labels=['Pendengar Rendah','Pendengar Sedang','Pendengar Aktif'])

#Melakukan output
df_no_outlier

/var/folders/c0/6lpjqwcj69lfvqtx74z82b6h0000gn/T/ipykernel_1278/3846883812.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_outlier['age_group'] = pd.cut(df['age'], bins=[0,25,50,70], labels=['Teen','Adult','Old'])
/var/folders/c0/6lpjqwcj69lfvqtx74z82b6h0000gn/T/ipykernel_1278/3846883812.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_outlier['listening_group'] = pd.cut(df['listening_time'], bins=[0,100,200,300], labels=['Pendengar Rendah','Pendengar Sedang','Pendengar Aktif'])


,user_id,gender,age,country,subscription_type,listening_time,songs_played_per_day,skip_rate,device_type,ads_listened_per_week,offline_listening,is_churned,age_group,listening_group
1,2,Other,33,DE,Family,141,62,0.34,Web,0,1,0,Adult,Pendengar Sedang
2,3,Male,38,AU,Premium,199,38,0.04,Mobile,0,1,1,Adult,Pendengar Sedang
3,4,Female,22,CA,Student,36,2,0.31,Mobile,0,1,0,Teen,Pendengar Rendah
4,5,Other,29,US,Family,250,57,0.36,Mobile,0,1,1,Adult,Pendengar Aktif
6,7,Female,39,UK,Premium,289,44,0.38,Desktop,0,1,0,Adult,Pendengar Aktif
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7995,7996,Other,44,DE,Student,237,36,0.30,Mobile,0,1,1,Adult,Pendengar Aktif
7996,7997,Male,34,AU,Premium,61,64,0.59,Mobile,0,1,0,Adult,Pendengar Rendah
7997,7998,Female,17,US,Free,81,62,0.33,Desktop,5,0,0,Teen,Pendengar Rendah
7998,7999,Female,34,IN,Student,245,94,0.27,Desktop,0,1,0,Adult,Pendengar Aktif


In [33]:
#8. Grouping dan Aggregation

# Mencari jumlah listening_time berdasarkan kelompok usia
sum_songs_played_per_day = df_no_outlier.groupby('age_group')['songs_played_per_day'].sum().reset_index()
sum_songs_played_per_day

/var/folders/c0/6lpjqwcj69lfvqtx74z82b6h0000gn/T/ipykernel_1278/3495342021.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sum_songs_played_per_day = df_no_outlier.groupby('age_group')['songs_played_per_day'].sum().reset_index()


,age_group,songs_played_per_day
0,Teen,73557
1,Adult,176523
2,Old,67130


In [34]:
#8. Grouping dan Aggregation

# Mencari jumlah songs_played_per_day berdasarkan subscription_type disetiap negara
sum_songs_played = df_no_outlier.groupby(['age_group','subscription_type'])['songs_played_per_day'].sum().reset_index()
sum_songs_played

/var/folders/c0/6lpjqwcj69lfvqtx74z82b6h0000gn/T/ipykernel_1278/1646559352.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sum_songs_played = df_no_outlier.groupby(['age_group','subscription_type'])['songs_played_per_day'].sum().reset_index()


,age_group,subscription_type,songs_played_per_day
0,Teen,Family,21890
1,Teen,Free,4029
2,Teen,Premium,24630
3,Teen,Student,23008
4,Adult,Family,54945
5,Adult,Free,8051
6,Adult,Premium,58042
7,Adult,Student,55485
8,Old,Family,19296
9,Old,Free,3406


In [35]:
#Agregasi beberapa statistik sekaligus per kota
stat_kelompok_usia = df_no_outlier.groupby('age_group')['songs_played_per_day'].agg(['mean', 'min', 'max', 'count']).reset_index()
stat_kelompok_usia

/var/folders/c0/6lpjqwcj69lfvqtx74z82b6h0000gn/T/ipykernel_1278/4256923688.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  stat_kelompok_usia = df_no_outlier.groupby('age_group')['songs_played_per_day'].agg(['mean', 'min', 'max', 'count']).reset_index()


,age_group,mean,min,max,count
0,Teen,51.259233,1,99,1435
1,Adult,49.907549,1,99,3537
2,Old,49.910781,1,99,1345
